# PixelForge — CUDA Acceleration & Benchmark Notebook
This notebook compiles and validates the **PixelForge** high-performance bilinear upscaling CUDA kernel on a remote cloud GPU (Google Colab).

### Step 0: Ensure GPU Runtime is Selected
Go to **Runtime** → **Change runtime type** → Hardware accelerator: **GPU** (T4, V100, or A100).

In [ ]:
# 1. Environment Verification
!nvidia-smi
!nvcc --version
!cmake --version

### Step 1: Clone Repository / Verify Directory
If you pushed your repo to GitHub, replace `YOUR_REPOSITORY_URL` below.
If you uploaded files to Colab directly, skip the git clone and ensure you are in the PixelForge folder.

In [ ]:
# Clone repository from GitHub
!rm -rf Vision_Flux
!git clone https://github.com/prabhjotSingh-002/Vision_Flux.git
%cd Vision_Flux
!git checkout feature/cuda-upscale
!ls -la

### Step 2: Configure and Build with CUDA Enabled
We use `PIXELFORGE_ENABLE_CUDA=ON` to compile `src/upscale/upscale_cuda.cu` with `nvcc`.

In [ ]:
!cmake -S . -B build \
       -DPIXELFORGE_ENABLE_CUDA=ON \
       -DCMAKE_BUILD_TYPE=Release

!cmake --build build --parallel 2

### Step 3: Run Unit & Correctness Tests
This verifies dimension handling, boundary clamping, and confirms that **CPU vs CUDA output difference <= 1 intensity level** across all channels.

In [ ]:
!./build/upscale_tests

### Step 4: Upscale Sample Image with CUDA
Process sample image at 2× and 4× scale factors using the custom CUDA kernel.

In [ ]:
# Upscale with CUDA (2x & 4x)
!./build/pixelforge upscale samples/demo.png outputs/demo_cuda_2x.png --scale 2 --backend cuda
!./build/pixelforge upscale samples/demo.png outputs/demo_cuda_4x.png --scale 4 --backend cuda

# Downscale with CUDA (2x reduction = 1/2 size, 4x reduction = 1/4 size)
!./build/pixelforge downscale samples/demo.png outputs/demo_cuda_half.png --scale 2 --backend cuda
!./build/pixelforge downscale samples/demo.png outputs/demo_cuda_quarter.png --scale 4 --backend cuda


### Step 5: Run Benchmark Matrix
Benchmark across synthetic sizes (256x256, 512x512, 1024x1024) and sample images. Measures CPU compute, GPU kernel time (via CUDA events), and GPU end-to-end time.

In [ ]:
!./build/pixelforge benchmark --scale 2 --warmup 5 --runs 25 --csv benchmarks/colab_results.csv
!./build/pixelforge benchmark --scale 4 --warmup 5 --runs 25 --csv benchmarks/colab_results.csv

### Step 6: Generate Performance Visualizations

In [ ]:
!python scripts/plot_benchmarks.py benchmarks/colab_results.csv benchmarks/colab_benchmark_plot.png

from IPython.display import Image, display
display(Image('benchmarks/colab_benchmark_plot.png'))